# PI Few-Shot Run Evaluation

Notebook skeleton for evaluating a trained model with the run-centric experiment layout. Everything is saved under one folder:

```text
experiments/runs/<model_run_id>/
```

In [ ]:
from __future__ import annotations

from pathlib import Path
import subprocess
import sys

from IPython.display import Image, Markdown, display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

PYTHON = sys.executable
RUNS_ROOT = PROJECT_ROOT / "experiments" / "runs"

def run_script(script: str, *args: str | Path) -> None:
    command = [PYTHON, script, *[str(arg) for arg in args]]
    print("$", " ".join(command))
    result = subprocess.run(command, cwd=PROJECT_ROOT, text=True, capture_output=True)
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)
    result.check_returncode()


def show_png(path: Path) -> None:
    display(Markdown(f"**{path.relative_to(PROJECT_ROOT)}**"))
    display(Image(filename=str(path)))

## Configure The Run

`MODEL_RUN_ID` is stable. Edit labels later in `experiments/runs/<model_run_id>/run.json` if plot names need to change.

In [ ]:
# Identity of this trained run. It determines the output folder
MODEL_RUN_ID = "20260530_raw_csi_proto_domain_separated"

# Tells the script how to instantiate the model (architecture loader)
MODEL_KEY = "raw_csi_proto"

# Human-readable label for run.json
# LABEL = "Raw CSI proto, domain-separated episodes"

# Only needed if checkpoint, training history or training curves are not in the default location
# CHECKPOINT = PROJECT_ROOT / "experiments" / "few_shot_raw_csi_proto_evaluation" / "raw_csi_proto_vs_doppler_featuremap_proto_20260529_193154" / "proto_model_best.pt"
# TRAINING_HISTORY = PROJECT_ROOT / "experiments" / "few_shot_raw_csi_proto_evaluation" / "raw_csi_proto_vs_doppler_featuremap_proto_20260529_193154" / "proto_training_history.json"
# TRAINING_CURVES = PROJECT_ROOT / "experiments" / "few_shot_raw_csi_proto_evaluation" / "raw_csi_proto_vs_doppler_featuremap_proto_20260529_193154" / "raw_csi_proto_training_curves.png"

BASELINE_RUN_IDS = [
    "raw_csi_mixed_proto",
    "doppler_featuremap_proto",
    "doppler_softmax_baseline",
    "doppler_pooled_head_proto",
]
UMAP_BASELINE_RUN_IDS = ["raw_csi_mixed_proto", "doppler_featuremap_proto"]

# assert CHECKPOINT.exists(), CHECKPOINT

## Evaluate And Plot

This computes K-shot records, copies training artifacts if provided, extracts UMAP, and writes comparison plots inside this run folder.

In [ ]:
args = [
    "--model-run-id", MODEL_RUN_ID,
    "--model-key", MODEL_KEY,
    # "--label", LABEL,
    # "--checkpoint", CHECKPOINT,
    # "--training-objective", "prototypical",
    # "--episode-style", "domain-separated",
    "--baseline-run-ids", *BASELINE_RUN_IDS,
    "--umap-baseline-run-ids", *UMAP_BASELINE_RUN_IDS,
    "--compute-umap",
    "--plot-kshot",
    "--plot-umap",
]
# if TRAINING_HISTORY.exists():
#     args += ["--training-history", TRAINING_HISTORY]
# if TRAINING_CURVES.exists():
#     args += ["--training-curves", TRAINING_CURVES]

run_script("scripts/evaluate_run.py", *args)

## Show Outputs

In [ ]:
run_dir = RUNS_ROOT / MODEL_RUN_ID
for png in sorted((run_dir / "comparisons").rglob("*.png")):
    show_png(png)

display(Markdown("### Run folder"))
display(Markdown(f"`{run_dir.relative_to(PROJECT_ROOT)}`"))